# Homework 10 - ST - 554 Big data analysis

## Author: Yefrid Cordoba

### Part 1

First we need to Setup a data stream using the `"rate"` format.\
Also we import the required functions to complete this part (`sqrt()`, `pmod()`)

In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import pmod, sqrt
spark = SparkSession.builder.getOrCreate()
rateDF = spark.readStream.format("rate").load()

Now we create a query that is going to take the value generated from the rate streaming data and calculate the squared root and the mod 4, and put those values in two new columns in the dataframe that is being generated.

In [12]:
rateDF = rateDF.withColumn("sqrt", sqrt(rateDF.value))\
                .withColumn("mod4", pmod(rateDF.value, 4))

We check that the dataframe that we get as output after start the stream is the expected, including the two new rows generated.

In [13]:
rateDF

DataFrame[timestamp: timestamp, value: bigint, sqrt: double, mod4: bigint]

As the expected dataframe is correct, we start the streaming data, we let it run for about 30 seconds and then stop it.

In [16]:
new_stm = rateDF\
            .writeStream\
            .outputMode("append")\
            .format("memory")\
            .queryName("stm_data")\
            .start()

26/04/20 16:37:49 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-6b63e08e-bfd5-4687-a073-21febf18a49b. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/20 16:37:49 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


After the selected time for run, we execute the next code chunck to stop the values generation. 

In [17]:
new_stm.stop()

26/04/20 16:38:34 WARN DAGScheduler: Failed to cancel job group c363c6a3-71bf-490d-8b34-42f41f0d4a5f. Cannot find active jobs for it.
26/04/20 16:38:34 WARN DAGScheduler: Failed to cancel job group c363c6a3-71bf-490d-8b34-42f41f0d4a5f. Cannot find active jobs for it.


And we can check the data that was stored in the memory during the execution of the code.

In [21]:
spark.sql("SELECT * FROM stm_data").show()

+--------------------+-----+------------------+----+
|           timestamp|value|              sqrt|mod4|
+--------------------+-----+------------------+----+
|2026-04-20 16:37:...|    0|               0.0|   0|
|2026-04-20 16:37:...|    1|               1.0|   1|
|2026-04-20 16:37:...|    2|1.4142135623730951|   2|
|2026-04-20 16:37:...|    3|1.7320508075688772|   3|
|2026-04-20 16:37:...|    4|               2.0|   0|
|2026-04-20 16:37:...|    5|  2.23606797749979|   1|
|2026-04-20 16:37:...|    6| 2.449489742783178|   2|
|2026-04-20 16:37:...|    7|2.6457513110645907|   3|
|2026-04-20 16:37:...|    8|2.8284271247461903|   0|
|2026-04-20 16:37:...|    9|               3.0|   1|
|2026-04-20 16:38:...|   10|3.1622776601683795|   2|
|2026-04-20 16:38:...|   11|   3.3166247903554|   3|
|2026-04-20 16:38:...|   12|3.4641016151377544|   0|
|2026-04-20 16:38:...|   13| 3.605551275463989|   1|
|2026-04-20 16:38:...|   14|3.7416573867739413|   2|
|2026-04-20 16:38:...|   15| 3.872983346207417

We can see that everything worked fine during the streaming of the data for the new columns that were created.

### Part 2